# SGMA-Net Loss Ablation Study on Kaggle (UIEB 800 train + UIEB 90 & EUVP test)

This notebook provides a complete environment for running systematic loss ablations on **SGMA-Net**:
- **Train dataset**: UIEB (strictly limited to **800 paired training images**)
- **Test datasets**: UIEB (**90 test images**) + EUVP (**`test_samples`**)
- **Base Loss (Loss gốc)**: $L_1$ + $L_{\text{perc}}$ (1.0 : 1.0)
- **Loss Ablations (Easily toggle 0 / 1)**:
  1. `(x1 T4)` Base Loss + Total Variation (TV) Loss
  2. `(x2 T4)` Base Loss + Total Variation (TV) Loss *(Benchmarking 1 vs 2 GPU speed)*
  3. `(x2 T4)` Base Loss + Edge Loss (Gaussian-Laplacian Pyramid)
  4. `(x2 T4)` Base Loss + Local Variance Loss (MobileIE, Yan et al. 2025)
  5. `(x2 T4)` Base Loss + UIQM Loss (Differentiable Underwater Image Quality Metric)

In [ ]:
import sys
import os
import subprocess
from pathlib import Path

IS_KAGGLE = Path("/kaggle").exists()
print(f"Running on: {'Kaggle' if IS_KAGGLE else 'Local Machine'}")

# Install essential lightweight requirements
!{sys.executable} -m pip install -q thop kornia

if IS_KAGGLE:
    REPO_DIR = Path("/kaggle/working/underwater-image-enhancement")
    if not REPO_DIR.exists():
        print("Cloning repository branch hungpt19...")
        subprocess.run([
            "git", "clone", "--branch", "hungpt19", "--single-branch", "--depth", "1",
            "https://github.com/heniath/underwater-image-enhancement.git", str(REPO_DIR)
        ], check=True)
    os.chdir(REPO_DIR)
    !{sys.executable} -m pip install -q -e .
else:
    REPO_DIR = Path.cwd()
    if not (REPO_DIR / "src" / "uwir").exists() and (REPO_DIR.parent / "src" / "uwir").exists():
        REPO_DIR = REPO_DIR.parent
    os.chdir(REPO_DIR)
    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))

print(f"Active Directory: {Path.cwd()}")

In [ ]:
# Run verification of all loss definitions against original papers and check gradient flow
!{sys.executable} scripts/verify_losses_against_papers.py
!{sys.executable} -m pytest tests/test_losses_ablation.py -v


In [ ]:
import torch
from pathlib import Path

# 1. GPU Detection
NUM_GPUS = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Detected GPUs : {NUM_GPUS}")
for i in range(NUM_GPUS):
    print(f"  [GPU {i}]: {torch.cuda.get_device_name(i)}")

# 2. Dataset Detection
def find_dataset(patterns, default_path):
    for p in patterns:
        matches = list(Path("/kaggle/input").glob(p)) if IS_KAGGLE else []
        if matches and matches[0].exists():
            return matches[0]
    if Path(default_path).exists():
        return Path(default_path)
    return Path(default_path)

UIEB_DIR = find_dataset(["**/UIEB*", "**/uieb*", "**/raw-890/.."], REPO_DIR / "datasets/UIEB")
EUVP_DIR = find_dataset(["**/EUVP*", "**/euvp*", "**/test_samples/.."], REPO_DIR / "datasets/EUVP")

print(f"UIEB Path : {UIEB_DIR}")
print(f"EUVP Path : {EUVP_DIR}")

### Bảng Cấu Hình Ablation Loss (Thay đổi 0 hoặc 1 để bật/tắt loss)

Bạn có thể dễ dàng sửa đổi các giá trị `0` và `1` trong danh sách `EXPERIMENTS` bên dưới để thử nghiệm các tổ hợp loss bất kỳ:
- `use_l1`: 1 (Loss L1 - MAE)
- `use_perc`: 1 (Loss Perceptual VGG-16, tỉ lệ 1:1 với L1)
- `use_tv`: Total Variation loss (mặc định trọng số: 0.001)
- `use_edge`: Edge loss (mặc định trọng số: 0.1)
- `use_lvw`: MobileIE Local Variance loss (mặc định trọng số: 0.1)
- `use_uiqm`: Differentiable UIQM loss (mặc định trọng số: 0.05)
- `num_gpus`: Số lượng GPU (1 hoặc 2) để so sánh tốc độ huấn luyện.

In [ ]:
# =============================================================================
# CONFIGURATION MATRIX (EASILY TOGGLE 0 OR 1)
# =============================================================================
EXPERIMENTS = [
    {
        "name": "sgmanet_5ch_base_tv_1gpu",
        "desc": "(x1 T4) Loss gốc (L1+Perc) + TV Loss",
        "num_gpus": 1,
        "use_l1": 1, "use_perc": 1, "use_tv": 1, "use_edge": 0, "use_lvw": 0, "use_uiqm": 0,
    },
    {
        "name": "sgmanet_5ch_base_tv_2gpu",
        "desc": "(x2 T4) Loss gốc (L1+Perc) + TV Loss (Speed comparison)",
        "num_gpus": 2,
        "use_l1": 1, "use_perc": 1, "use_tv": 1, "use_edge": 0, "use_lvw": 0, "use_uiqm": 0,
    },
    {
        "name": "sgmanet_5ch_base_edge_2gpu",
        "desc": "(x2 T4) Loss gốc (L1+Perc) + Edge Loss",
        "num_gpus": 2 if NUM_GPUS >= 2 else 1,
        "use_l1": 1, "use_perc": 1, "use_tv": 0, "use_edge": 1, "use_lvw": 0, "use_uiqm": 0,
    },
    {
        "name": "sgmanet_5ch_base_lvw_2gpu",
        "desc": "(x2 T4) Loss gốc (L1+Perc) + Local Variance Loss (MobileIE)",
        "num_gpus": 2 if NUM_GPUS >= 2 else 1,
        "use_l1": 1, "use_perc": 1, "use_tv": 0, "use_edge": 0, "use_lvw": 1, "use_uiqm": 0,
    },
    {
        "name": "sgmanet_5ch_base_uiqm_2gpu",
        "desc": "(x2 T4) Loss gốc (L1+Perc) + UIQM Loss",
        "num_gpus": 2 if NUM_GPUS >= 2 else 1,
        "use_l1": 1, "use_perc": 1, "use_tv": 0, "use_edge": 0, "use_lvw": 0, "use_uiqm": 1,
    },
]

# Common training hyper-parameters
EPOCHS = 100         # Sửa thành 1 nếu muốn chạy smoke test nhanh
BATCH_SIZE = 16
CROP_SIZE = 256
LR = 1e-4
MODEL_VARIANT = "sgmanet_5ch"
UIEB_LIMIT = 800     # Huấn luyện đúng 800 ảnh UIEB

print(f"Loaded {len(EXPERIMENTS)} ablation experiments for execution.")

In [ ]:
import time
import gc

timing_results = {}

for idx, exp in enumerate(EXPERIMENTS, start=1):
    run_name = exp["name"]
    gpus = exp["num_gpus"]
    print(f"\n{'='*70}")
    print(f" [Run {idx}/{len(EXPERIMENTS)}] Starting: {exp['desc']}")
    print(f" Run Name: {run_name} | GPUs: {gpus}")
    print(f"{'='*70}")

    cmd = [
        sys.executable, "-m", "uwir.cli.train",
        "--model", MODEL_VARIANT,
        "--dataset", "uieb",
        "--data_train_uieb", str(UIEB_DIR),
        "--uieb_limit", str(UIEB_LIMIT),
        "--run_name", run_name,
        "--nEpochs", str(EPOCHS),
        "--batchSize", str(BATCH_SIZE),
        "--cropSize", str(CROP_SIZE),
        "--lr", str(LR),
        "--num_gpus", str(gpus),
        "--use_l1", str(exp["use_l1"]),
        "--use_perc", str(exp["use_perc"]),
        "--use_tv", str(exp["use_tv"]),
        "--use_edge", str(exp["use_edge"]),
        "--use_lvw", str(exp["use_lvw"]),
        "--use_uiqm", str(exp["use_uiqm"]),
    ]

    t_start = time.time()
    subprocess.run(cmd, check=True)
    t_end = time.time()
    elapsed = t_end - t_start
    timing_results[run_name] = elapsed
    print(f">> Finished {run_name} in {elapsed:.1f}s ({elapsed/60:.2f} min)")

    # Clean VRAM & cache before next run
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    time.sleep(3)

In [ ]:
print("\n=================================================================")
print(" Evaluating All Checkpoints on Combined Test Set (UIEB-90 + EUVP)")
print("=================================================================")

eval_cmd = [
    sys.executable, "-m", "uwir.cli.evaluate",
    "--checkpoint_dir", "./checkpoints",
    "--eval_benchmark", "uieb+euvp",
    "--data_train_uieb", str(UIEB_DIR),
    "--data_train_euvp", str(EUVP_DIR),
    "--val_folder", "./results/eval_loss_ablations"
]
subprocess.run(eval_cmd, check=True)

In [ ]:
import json
import pandas as pd

res_file = Path("./results/eval_loss_ablations/test_results_all.json")
if res_file.exists():
    with open(res_file) as f:
        data = json.load(f)
    rows = []
    for run_name, info in data.items():
        tm = info.get("test_metrics") or {}
        rows.append({
            "Experiment": run_name,
            "PSNR (dB)": round(tm.get("psnr", 0.0), 2),
            "SSIM": round(tm.get("ssim", 0.0), 4),
            "CIEDE2000": round(tm.get("ciede2000", 0.0), 2),
            "UCIQE": round(tm.get("uciqe", 0.0), 4),
            "UIQM": round(tm.get("uiqm", 0.0), 4),
            "Time (min)": round(timing_results.get(run_name, 0.0) / 60, 2),
        })
    df = pd.DataFrame(rows)
    print("\n--- BẢNG TỔNG HỢP KẾT QUẢ ABLATION STUDY LOSS ---")
    display(df) if "display" in globals() else print(df.to_string())
else:
    print("[WARN] test_results_all.json not found yet.")